# 订单簿重建：综合演示

这个 Notebook 只用来**看**：所有功能都在 `src/` 的三个 `.py` 文件里，这里只调用函数、打印、画图。

整体思路分两步：
1. **拼逐笔流**：把 `orders.csv`（逐笔委托）和 `trades.csv`（逐笔成交）整理成同一张十列的表，排好先后。两个交易所的原始数据不一样，各写各的：
   - `sz_stock_build_orderbook.py`：深交所
   - `sh_stock_build_orderbook.py`：上交所
2. **建订单簿**：沿着逐笔流一行一行回放，每一行改一次订单簿，最后和交易所快照对账。这一步深沪共用：`build_orderbook.py`

目录：
- 第一部分：深市，拼逐笔流（000001）
- 第二部分：沪市，拼逐笔流（603353）
- 第三部分：订单簿（回放、某一时刻的盘口、收盘对账）

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from build_orderbook import build_order_book, compare_with_snapshot, check_tick_by_tick
from sz_stock_build_orderbook import read_orders, read_trades, load_sz_tick_by_tick, count_by_session
from sh_stock_build_orderbook import (
    read_sh_orders, read_sh_trades,
    sh_orders_to_tick_by_tick, sh_trades_to_tick_by_tick,
    add_sort_columns, load_sh_tick_by_tick,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Mac 上显示中文
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "Heiti SC"]
plt.rcParams["axes.unicode_minus"] = False

DATA = "../../data/"   # 数据在 L2 数据/data/ 下
SZ_FOLDER = DATA + "SZ_000001_20260302"
SH_FOLDER = DATA + "SH_603353_20260302"

---
# 第一部分：深市，拼逐笔流（000001）

## 1.1 原始数据

深市的两个文件共用一套 `seq`（交易所发的排队号），撤单写在成交文件里：被撤的一边写订单号，另一边是 0，价格是 0。

In [ ]:
sz_orders = read_orders(SZ_FOLDER)
sz_trades = read_trades(SZ_FOLDER)
print("委托文件：")
display(sz_orders.head(3))
print("成交文件里的成交和撤单：")
display(sz_trades[sz_trades["event"] == "TRADE"].head(2))
display(sz_trades[sz_trades["event"] == "CANCEL"].head(2))

## 1.2 拼成逐笔流

先各自整理成同样的十列，再拼，再按 `seq` 排。验收数字：总行数 151,713；ADD / TRADE / CANCEL = 81,113 / 45,405 / 25,195；LIMIT / MARKET / BEST_OWN = 80,025 / 929 / 159；seq 不重复；时间倒退 0 次。

In [ ]:
sz_tick_by_tick = load_sz_tick_by_tick(SZ_FOLDER)
display(sz_tick_by_tick.head(10))
check_tick_by_tick(sz_tick_by_tick)

In [ ]:
count_by_session(sz_tick_by_tick)

## 1.3 每分钟的逐笔条数

开盘和收盘最密集，午休是空的。

In [ ]:
minute = sz_tick_by_tick["time"] // 100000          # hhmmssmmm -> hhmm
table = sz_tick_by_tick.groupby([minute, "event"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 4))
x = range(len(table))
for name in ["ADD", "TRADE", "CANCEL"]:
    ax.plot(x, table[name], label=name, linewidth=1)
ticks = list(range(0, len(table), 15))
ax.set_xticks(ticks)
ax.set_xticklabels([f"{table.index[i] // 100}:{table.index[i] % 100:02d}" for i in ticks], rotation=45)
ax.set_ylabel("每分钟条数")
ax.set_title("000001 每分钟逐笔条数")
ax.legend()
plt.tight_layout()
plt.show()

## 1.4 跟踪一笔委托

挑一笔成交里的买方订单，看它在逐笔流里出现的所有行：先 ADD 挂上去，再被成交、撤单扣掉。

In [ ]:
trade = sz_tick_by_tick[sz_tick_by_tick["event"] == "TRADE"].iloc[1000]
no = trade["buy_no"]
life = sz_tick_by_tick[
    (sz_tick_by_tick["order_no"] == no) | (sz_tick_by_tick["buy_no"] == no) | (sz_tick_by_tick["sell_no"] == no)
]
life

---
# 第二部分：沪市，拼逐笔流（603353）

## 2.1 原始数据

和深市比，注意三件事：
- `seq` 是空的：沪市没有委托和成交共用的排队号；
- `order_type` 是空的：万得沪市不提供委托类型；
- 撤单在委托文件里（CANCEL，`order_no` 是被撤订单的号），还有 STATUS（交易阶段切换）。

In [ ]:
orders = read_sh_orders(SH_FOLDER)
display(orders.head(6))
display(orders["event"].value_counts())
display(orders[orders["event"] == "STATUS"])

In [ ]:
trades = read_sh_trades(SH_FOLDER)
display(trades.head(5))

## 2.2 各自整理成十列

委托这边：STATUS 丢掉；ADD 当限价；CANCEL 装扮成深市撤单的样子（被撤的订单号放进 `buy_no` 或 `sell_no`，另一边填 0）。`seq` 先空着。

In [ ]:
order_part = sh_orders_to_tick_by_tick(orders)
print("ADD 行：")
display(order_part[order_part["event"] == "ADD"].head(3))
print("CANCEL 行：")
display(order_part[order_part["event"] == "CANCEL"].head(3))

In [ ]:
trade_part = sh_trades_to_tick_by_tick(trades)
display(trade_part.head(3))

## 2.3 找一个场景：先成交、后 ADD

（这一格只是找例子用，不是功能代码。）

要找的是：某张订单一进门先当主动方成交，剩下的部分在**同一毫秒**里以 ADD 出现。
1. 每笔成交取较大的订单号，就是主动方（后到的），加一列 `agg_no`；
2. 拿“主动方订单号 + 时间”去 ADD 里查，查得到的就是；
3. 只看 9:30 以后（集合竞价没有“一进门就吃单”）；
4. 取最早的一个，把那一毫秒的原始数据打印出来。

In [ ]:
adds = orders[orders["event"] == "ADD"]
trades_with_agg = trades.assign(agg_no=trades[["buy_no", "sell_no"]].max(axis=1))
matched = trades_with_agg.merge(
    adds[["order_no", "time"]], left_on=["agg_no", "time"], right_on=["order_no", "time"]
)
matched = matched[matched["time"] >= 93000000]
example_no = int(matched["agg_no"].iloc[0])
example_time = int(matched["time"].iloc[0])
print("例子：订单", example_no, "，时间", example_time)

print("\n原始委托文件里，这一毫秒：")
display(orders[orders["time"] == example_time])
print("原始成交文件里，这一毫秒：")
display(trades[trades["time"] == example_time])

## 2.4 只按 time 排会出什么问题

直接拼起来只按 `time` 排，同一毫秒分不出先后：委托的行排在成交前面，于是这张订单的 ADD（剩余量）排在了它自己的成交前面。回放时会先挂上剩余量，再被成交扣一次，剩余量变成负数。

In [ ]:
merged = pd.concat([order_part, trade_part], ignore_index=True)
only_time = merged.sort_values("time", kind="stable", ignore_index=True)
display(only_time[only_time["time"] == example_time])

## 2.5 加 key 和 rank，再排序

- `key`：这一行在“订单号这条线”上的位置。ADD 用自己的订单号；成交用较大的订单号（主动方）；撤单用被撤的订单号。
- `rank`：key 相同时谁先，数字小的在前。股票 / ETF 是成交 0、ADD 1、撤单 2。

按（time, key, rank）逐级比较：time 相同才比 key，key 也相同才比 rank。

In [ ]:
with_keys = add_sort_columns(merged, trade_before_add=True)
same_ms = with_keys[with_keys["time"] == example_time]
display(same_ms.sort_values(["time", "key", "rank"], kind="stable"))

完整的合并就是 `load_sh_tick_by_tick`：排好后从 1 开始编 `seq`，去掉辅助列，得到和深市一样的十列逐笔流。

股票、ETF 用默认的 `trade_before_add=True`；可转债必须传 `False`。

In [ ]:
sh_tick_by_tick = load_sh_tick_by_tick(SH_FOLDER, trade_before_add=True)
display(sh_tick_by_tick[sh_tick_by_tick["time"] == example_time])
check_tick_by_tick(sh_tick_by_tick)

---
# 第三部分：订单簿

订单簿就是三本字典：
- `bids` / `asks`：**黑板**，价格 → 该价位总量，就是大家看到的盘口；
- `orders`：**本子**，订单号 → [方向, 价格, 剩余量]。成交、撤单的记录里没有订单当初的挂单价，只能靠本子查。

## 3.1 停在某一刻：10:00 的盘口

`stop_time` 让回放停在任意时刻，手上的订单簿就是那一刻的。

In [ ]:
book_10 = build_order_book(sz_tick_by_tick, stop_time=100000000)
bids, asks = book_10.top_levels(10)
print("买一 / 卖一：", book_10.best_bid(), "/", book_10.best_ask())
print("本子里还挂着的订单数：", len(book_10.orders))
display(pd.DataFrame(bids, columns=["买价", "买量"]))
display(pd.DataFrame(asks, columns=["卖价", "卖量"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([p / 10000 for p, _ in bids], [q for _, q in bids], height=0.008, color="tab:red", label="买盘")
ax.barh([p / 10000 for p, _ in asks], [q for _, q in asks], height=0.008, color="tab:green", label="卖盘")
ax.set_xlabel("数量（股）")
ax.set_ylabel("价格（元）")
ax.set_title("000001 10:00 前十档")
ax.legend()
plt.tight_layout()
plt.show()

## 3.2 收盘对账：深市

全天回放后，前十档和交易所最后一张快照逐档比。深市三项异常计数都应该是 0。

In [ ]:
sz_book = build_order_book(sz_tick_by_tick)
sz_snapshots = pd.read_csv(SZ_FOLDER + "/snapshots.csv")
display(compare_with_snapshot(sz_book, sz_snapshots.iloc[-1]))
print(sz_book.problems)

## 3.3 收盘对账：沪市三个品种

沪市主要看两样：**剩余量变负 = 0，十档全部一致**。“找不到订单号”大部分是幽灵订单（一进门就吃完、没有 ADD），属于正常现象。可转债 110093 的 ADD 是完整量，要传 `trade_before_add=False`。

In [ ]:
samples = [
    ("SH_603353_20260302", True),    # 股票
    ("SH_518880_20260302", True),    # ETF（数据量大，要跑一会儿）
    ("SH_110093_20260302", False),   # 可转债
]
rows = []
for name, trade_before_add in samples:
    folder = DATA + name
    book = build_order_book(load_sh_tick_by_tick(folder, trade_before_add))
    table = compare_with_snapshot(book, pd.read_csv(folder + "/snapshots.csv").iloc[-1])
    rows.append({"数据": name, "trade_before_add": trade_before_add,
                 "十档一致": int(table["是否一致"].sum()), **book.problems})
pd.DataFrame(rows)